<a href="https://colab.research.google.com/github/exdsgift/NerGuard/blob/main/PII_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Importing the modules

In [ ]:
!pip install -q "transformers>=4.40" datasets seqeval accelerate

import torch
import os, json, unicodedata
from datasets import load_dataset, DatasetDict
import numpy as np
from typing import List, Dict, Tuple
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from seqeval.metrics import f1_score, classification_report
from seqeval.scheme import IOB2
import pprint as pp

#Specifying CUDA as the device for torch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
!nvidia-smi

# Loading dataset

In [ ]:
ds = load_dataset("gretelai/synthetic_pii_finance_multilingual")
print(ds)

In [ ]:
if "validation" not in ds:
    tmp = ds["train"].train_test_split(test_size=0.1, seed=42)
    ds = DatasetDict({
        "train": ds["train"],
        "validation": tmp["test"],
        "test": tmp["train"]
    })
print(ds)

In [ ]:
ds.shape

In [ ]:
df = ds["train"].to_pandas()
df.head()

#Activating the mdeberta-v3-base tokenizer (fast) and support functions

In [ ]:
try:
  tok = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base", use_fast=True)
  assert tok.is_fast, "Serve tokenizer fast per offset_mapping"
except Exception as e:
  print("An error occurred while downloading the tokenizer.")
  print(str(e))
  import traceback
  print(traceback.format_exc())

In [ ]:
def parse_spans(spans):
    if isinstance(spans, str):
        try:
            spans = json.loads(spans)
        except json.JSONDecodeError:
            return []
    if not isinstance(spans, list):
        return []
    out = []
    for s in spans:
        if isinstance(s, dict) and all(k in s for k in ("start","end","label")):
            out.append(s)
    return out

In [ ]:
def collect_label_set(dataset_dict):
    labels = set()
    for split in dataset_dict.keys():
        for ex in dataset_dict[split]:
            for s in parse_spans(ex["pii_spans"]):
                labels.add(s["label"])
    return sorted(labels)

entity_labels = collect_label_set(ds)  # es: ['company','date','name','street_address', ...]
bio_labels = ["O"] + [f"B-{l}" for l in entity_labels] + [f"I-{l}" for l in entity_labels]
label2id = {l:i for i,l in enumerate(bio_labels)}
id2label = {i:l for l,i in label2id.items()}

In [ ]:
bio_labels

spans → BIO with offset_mapping

In [ ]:
def spans_to_bio(text, spans, label2id, tokenizer, max_length=256):
    import unicodedata
    text = unicodedata.normalize("NFC", text)
    spans = parse_spans(spans)

    # 1) maschera char-level
    char_tags = ["O"] * len(text)
    for s in spans:
        start, end, lab = s.get("start"), s.get("end"), s.get("label")
        if not isinstance(start,int) or not isinstance(end,int) or not isinstance(lab,str):
            continue
        if start < 0 or end <= start or end > len(text):
            continue
        if f"B-{lab}" not in label2id or f"I-{lab}" not in label2id:
            continue
        char_tags[start] = f"B-{lab}"
        for i in range(start+1, end):
            char_tags[i] = f"I-{lab}"

    # 2) tokenizza con special e offsets
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        return_attention_mask=True,
        add_special_tokens=True,     # <-- importante
        truncation=True,
        max_length=max_length
    )
    offsets = enc["offset_mapping"]

    tags = []
    for (a, b) in offsets:
        if a == 0 and b == 0:
            tags.append(None)        # special token -> ignora in loss
            continue
        if a == b:
            tags.append("O")
            continue
        start_tag = char_tags[a] if 0 <= a < len(text) else "O"
        if start_tag.startswith("B-"):
            lab = start_tag
        else:
            window = [t for t in char_tags[a:b] if t != "O"]
            if window:
                any_lab = window[0]
                lab = "I-" + any_lab.split("-", 1)[1]
            else:
                lab = "O"
        tags.append(lab)

    ignore_index = -100
    label_ids = [ignore_index if t is None else label2id[t] for t in tags]
    return enc["input_ids"], enc["attention_mask"], label_ids


Processing all splits

In [ ]:
from transformers import DataCollatorForTokenClassification

TEXT_COL = "generated_text"
SPAN_COL = "pii_spans"

def preprocess_batch(batch):
    input_ids, attention_mask, labels = [], [], []
    for text, spans in zip(batch[TEXT_COL], batch[SPAN_COL]):
        ids, mask, y = spans_to_bio(text, spans, label2id, tok, max_length=256)
        input_ids.append(ids); attention_mask.append(mask); labels.append(y)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

remove_cols = list(ds["train"].column_names)
processed = DatasetDict()
for split in ds.keys():
    processed[split] = ds[split].map(
        preprocess_batch,
        batched=True,
        remove_columns=remove_cols
    ).with_format("torch",
                  columns=["input_ids","attention_mask"],
                  output_all_columns=True)

#.with_format("torch", columns=["input_ids","attention_mask","labels"])
train_processed = processed["train"]
val_processed   = processed["validation"]

data_collator = DataCollatorForTokenClassification(
    tokenizer=tok,
    padding=True,
    label_pad_token_id=-100
)

In [ ]:
ds = train_processed.to_pandas()
ds.head(3)

In [ ]:
print(train_processed[0])

In [ ]:
from pprint import pprint

print("num_labels:", len(label2id))
pprint(label2id)           # label string -> id
# If you also want the reverse:
id2label = {i:l for l,i in label2id.items()}


In [ ]:
# take one example from the processed train set
ex = train_processed[0]

tokens = tok.convert_ids_to_tokens(ex["input_ids"])
tag_ids = ex["labels"]
tag_str = [id2label[int(i)] for i in tag_ids if i != -100]
aligned_data = [(tokens[i], tag_ids[i], id2label[int(tag_ids[i])]) for i in range(len(tokens)) if tag_ids[i] != -100]

for t, tid, ts in aligned_data:
    print(f"{t:20s}  {tid:3d}  {ts}")

#Selecting a batch size and creating an iterator

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from keras.utils import pad_sequences
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertConfig
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
from transformers import BertForSequenceClassification
from tqdm import tqdm, trange  #for progress bars
import pandas as pd
import io
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image #for image rendering

In [ ]:
train_processed

In [ ]:
# Select a batch size for training. For fine-tuning BERT on a specific task, the authors recommend a batch size of 16 or 32
batch_size = 32

# Create an iterator of our data with torch DataLoader. This helps save on memory during training because, unlike a for loop,
# with an iterator the entire dataset does not need to be loaded into memory

# train_data = TensorDataset(train_processed['input_ids'], train_processed['attention_mask'], train_processed['labels'])
train_sampler = RandomSampler(train_processed)
train_dataloader = DataLoader(train_processed, sampler=train_sampler, batch_size=batch_size, collate_fn=data_collator)

# validation_data = TensorDataset(val_processed)
validation_sampler = SequentialSampler(val_processed)
validation_dataloader = DataLoader(val_processed, sampler=validation_sampler, batch_size=batch_size, collate_fn=data_collator)


In [ ]:
# Initializing a BERT bert-base-uncased style configuration
from transformers import AutoConfig, AutoModel, AutoTokenizer
model_name = "microsoft/deberta-v3-base"

configuration = AutoConfig.from_pretrained(model_name)

# Initializing a model from the bert-base-uncased style configuration
model = AutoModel.from_pretrained(model_name, config=configuration)

# Accessing the model configuration
# configuration = model.config
print(configuration)

In [ ]:
# @title
from collections import Counter

used_ids = Counter()
for example in train_processed:
    used_ids.update(i for i in example["labels"] if i != -100)   # ignore masked

print("Unique label IDs used:", len(used_ids))
for lid, cnt in used_ids.most_common()[:30]:
    print(f"{lid:3d}  {id2label[lid]:30s}  {cnt}")

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/mdeberta-v3-base",
    num_labels=len(bio_labels),
    id2label=id2label,
    label2id=label2id
)
model = nn.DataParallel(model)
model.to(device)

In [ ]:
# Parametri ottimizzati per DaBERTa v3 su task di NER/PII detection
param_optimizer = list(model.named_parameters())

# Non applicare weight decay a bias e LayerNorm
no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']

optimizer_grouped_parameters = [
    {
        'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
        'weight_decay': 0.01  # Valore tipico per DaBERTa (0.01-0.1)
    },
    {
        'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
        'weight_decay': 0.0
    }
]

In [ ]:
# Displaying a sample of the parameter_optimizer:  layer 3
layer_parameters = [p for n, p in model.named_parameters() if 'layer.3' in n]

In [ ]:
# Displaying names of parameters for which weight decay is not applied
no_decay

In [ ]:
# Displaying the list of the two dictionaries
small_sample = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)][:2],
     'weight_decay_rate': 0.01},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)][:2],
     'weight_decay_rate': 0.0}
]

for i, group in enumerate(small_sample):
    print(f"Group {i+1}:")
    print(f"Weight decay rate: {group['weight_decay_rate']}")
    for j, param in enumerate(group['params']):
        print(f"Parameter {j+1}: {param}")

#The hyperparameters for the training loop

In [ ]:
# Number of training epochs (authors recommend between 2 and 4)
epochs = 4

optimizer = AdamW(optimizer_grouped_parameters,
                  lr = 2e-5, # args.learning_rate - default range: 1e-5 a 5e-5
                  eps = 1e-8 # args.adam_epsilon  - default is 1e-8.
                  )
# Total number of training steps is number of batches * number of epochs.
# `train_dataloader` contains batched data so `len(train_dataloader)` gives
# us the number of batches.
total_steps = len(train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

In [ ]:
# Creating the Accuracy Measurement Function
# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def compute_metrics(preds, labels, label_list):
    """
    Metrics for PII detection (precision, recall, F1)
    """
    pred_flat = np.argmax(preds, axis=2).flatten()
    labels_flat = labels.flatten()

    # Rimuovi i padding tokens (assumendo che -100 sia usato per padding)
    mask = labels_flat != -100
    pred_flat = pred_flat[mask]
    labels_flat = labels_flat[mask]

    # F1 score (più importante dell'accuracy per PII detection!)
    f1 = f1_score(labels_flat, pred_flat, average='weighted')
    accuracy = np.sum(pred_flat == labels_flat) / len(labels_flat)

    return {
        'accuracy': accuracy,
        'f1': f1
    }

# training loop

In [ ]:
import gc
import torch

# Prima del training
gc.collect()
torch.cuda.empty_cache()

# Verifica memoria disponibile
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

In [ ]:
def estimate_gpu_memory(model, batch_size, seq_length, num_labels):
    """
    Stima memoria GPU necessaria (approssimativa)
    """
    # Parametri del modello
    num_params = sum(p.numel() for p in model.parameters())

    # Memoria per i parametri (in GB)
    # FP32 = 4 bytes per parametro
    model_memory = (num_params * 4) / (1024**3)

    # Memoria per i gradienti (stessa dimensione dei parametri)
    gradient_memory = model_memory

    # Memoria per optimizer state (Adam = 2x parametri)
    optimizer_memory = model_memory * 2

    # Memoria per attivazioni (molto approssimativa)
    # DaBERTa: ~12 * num_layers * batch_size * seq_length * hidden_size * 4 bytes
    hidden_size = model.module.config.hidden_size # Access config through .module
    num_layers = model.module.config.num_hidden_layers # Access config through .module
    activation_memory = (12 * num_layers * batch_size * seq_length * hidden_size * 4) / (1024**3)

    total = model_memory + gradient_memory + optimizer_memory + activation_memory

    print("="*60)
    print("STIMA MEMORIA GPU (approssimativa)")
    print("="*60)
    print(f"Modello:                {model_memory:.2f} GB")
    print(f"Gradienti:              {gradient_memory:.2f} GB")
    print(f"Optimizer (Adam):       {optimizer_memory:.2f} GB")
    print(f"Attivazioni (forward):  {activation_memory:.2f} GB")
    print(f"-" * 60)
    print(f"TOTALE STIMATO:         {total:.2f} GB")
    print(f"\nBatch size:             {batch_size}")
    print(f"Sequence length:        {seq_length}")
    print(f"Parametri totali:       {num_params:,}")
    print("="*60)

    return total

# Usa così:
estimate_gpu_memory(model, batch_size=16, seq_length=512, num_labels=len(label2id))

In [ ]:
import torch
import gc

def find_optimal_batch_size(model, device, max_seq_length=512, num_labels=None):
    """
    Trova il batch size ottimale tramite binary search
    """
    if num_labels is None:
        # Access config through .module for DataParallel
        num_labels = model.module.config.num_labels

    model.train()  # Importante: train mode usa più memoria

    def create_test_batch(batch_size):
        return {
            'input_ids': torch.randint(0, model.module.config.vocab_size, (batch_size, max_seq_length)).to(device),
            'attention_mask': torch.ones(batch_size, max_seq_length).to(device),
            'labels': torch.randint(-100, num_labels, (batch_size, max_seq_length)).to(device)
        }

    # Binary search - Start with a potentially smaller max_bs
    min_bs, max_bs = 1, 16  # Reduced initial max_bs
    optimal_bs = 1

    print("🔍 Cercando batch size ottimale...")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria totale: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"Sequence length: {max_seq_length}\n")

    while min_bs <= max_bs:
        batch_size = (min_bs + max_bs) // 2
        if batch_size == 0: # Avoid batch size 0
            break

        # Pulisci memoria
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        try:
            print(f"Testing batch_size={batch_size:2d}...", end=" ")

            batch = create_test_batch(batch_size)

            # Forward pass
            outputs = model(**batch)
            loss = outputs.loss

            # Backward pass (questo usa MOLTA più memoria!)
            loss.backward()

            # Misura memoria PEAK (massima usata)
            mem_allocated = torch.cuda.max_memory_allocated() / 1024**3
            mem_reserved = torch.cuda.max_memory_reserved() / 1024**3

            print(f"✅ OK | Peak: {mem_allocated:.2f}GB alloc, {mem_reserved:.2f}GB reserved")

            optimal_bs = batch_size
            min_bs = batch_size + 1 # Try larger batch size

        except RuntimeError as e:
            if "out of memory" in str(e).lower(): # Check for "out of memory" case-insensitively
                print(f"❌ OOM")
                max_bs = batch_size - 1 # Reduce batch size
            else:
                # Re-raise other RuntimeErrors
                raise e
        except Exception as e:
             # Catch any other exceptions during testing
             print(f"❌ Error: {e}")
             # Decide how to handle other errors, perhaps stop or reduce batch size
             max_bs = batch_size - 1 # Assuming other errors might also be memory related or blocking progress
        finally:
            # Cleanup
            if 'batch' in locals():
                del batch
            if 'outputs' in locals():
                del outputs
            if 'loss' in locals():
                del loss

            # Pulisci gradienti
            if model is not None:
                try:
                    model.zero_grad()
                except Exception as zero_grad_e:
                     print(f"Error during zero_grad: {zero_grad_e}")


            gc.collect()
            torch.cuda.empty_cache()

    print(f"\n{'='*70}")
    print(f"✨ RISULTATI:")
    print(f"{'='*70}")
    print(f"Batch size massimo testato con successo:              {optimal_bs}")
    print(f"Consigliato (safe, ~70%):        {max(1, int(optimal_bs * 0.7))}")
    print(f"Con gradient accumulation 2x:    batch={max(1, optimal_bs // 2)}, accum_steps=2")
    print(f"Con gradient accumulation 4x:    batch={max(1, optimal_bs // 4)}, accum_steps=4")
    print(f"Con gradient accumulation 8x:    batch={max(1, optimal_bs // 8)}, accum_steps=8")
    print(f"{'='*70}\n")

    return optimal_bs

# Esegui il test
gc.collect()
torch.cuda.empty_cache()

optimal_bs = find_optimal_batch_size(
    model,
    device,
    max_seq_length=512,
    num_labels=len(label2id)
)

In [ ]:
from sklearn.metrics import classification_report, f1_score
import numpy as np
import torch
from tqdm import trange

# Funzione metriche per NER
def compute_metrics(preds, labels):
    """
    Calcola metriche per PII detection (ignora padding -100)
    """
    pred_flat = np.argmax(preds, axis=2).flatten()
    labels_flat = labels.flatten()

    # Rimuovi padding
    mask = labels_flat != -100
    pred_flat = pred_flat[mask]
    labels_flat = labels_flat[mask]

    f1 = f1_score(labels_flat, pred_flat, average='weighted', zero_division=0)
    accuracy = np.sum(pred_flat == labels_flat) / len(labels_flat)

    return {'accuracy': accuracy, 'f1': f1}

# Store metriche
train_loss_set = []
val_loss_set = []
val_accuracy_set = []
val_f1_set = []

# Training loop
for epoch in trange(epochs, desc="Epoch"):

    # ============== TRAINING ==============
    model.train()

    tr_loss = 0
    nb_tr_steps = 0

    for step, batch in enumerate(train_dataloader):
        # ✅ Accedi ai dati come dizionario e sposta su GPU
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            b_input_ids,
            attention_mask=b_input_mask,
            labels=b_labels
        )

        loss = outputs.loss

        # Backward pass
        loss.backward()

        # Gradient clipping (importante!)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update
        optimizer.step()
        scheduler.step()

        # Tracking
        tr_loss += loss.item()
        nb_tr_steps += 1

        if step % 100 == 0 and step > 0:
            print(f"  Step {step}, Loss: {loss.item():.4f}")

    avg_train_loss = tr_loss / nb_tr_steps
    train_loss_set.append(avg_train_loss)
    print(f"\nEpoch {epoch + 1}")
    print(f"  Average training loss: {avg_train_loss:.4f}")


    # ============== VALIDATION ==============
    model.eval()

    eval_loss = 0
    nb_eval_steps = 0
    predictions, true_labels = [], []

    for batch in validation_dataloader:
        # ✅ Stesso pattern per validation
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        with torch.no_grad():
            outputs = model(
                b_input_ids,
                attention_mask=b_input_mask,
                labels=b_labels
            )

        loss = outputs.loss
        logits = outputs.logits

        eval_loss += loss.item()

        # Sposta su CPU
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.cpu().numpy()

        predictions.append(logits)
        true_labels.append(label_ids)
        nb_eval_steps += 1

    # Concatena batch
    predictions = np.concatenate(predictions, axis=0)
    true_labels = np.concatenate(true_labels, axis=0)

    # Calcola metriche
    metrics = compute_metrics(predictions, true_labels)

    avg_val_loss = eval_loss / nb_eval_steps
    val_loss_set.append(avg_val_loss)
    val_accuracy_set.append(metrics['accuracy'])
    val_f1_set.append(metrics['f1'])

    print(f"  Validation loss: {avg_val_loss:.4f}")
    print(f"  Validation Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Validation F1: {metrics['f1']:.4f}")
    print()

print("\n" + "="*50)
print("TRAINING COMPLETED")
print("="*50)

In [ ]:
t = []

# Store our loss and accuracy for plotting
train_loss_set = []

# trange is a tqdm wrapper around the normal python range
for _ in trange(epochs, desc="Epoch"):

  # Training

  # Set our model to training mode (as opposed to evaluation mode)
  model.train()

  # Tracking variables
  tr_loss = 0
  nb_tr_examples, nb_tr_steps = 0, 0

  # Train the data for one epoch
  for step, batch in enumerate(train_dataloader):
    # Add batch to GPU
    batch = tuple(t.to(device) for t in batch)
    # Unpack the inputs from our dataloader
    b_input_ids, b_input_mask, b_labels = batch
    # Clear out the gradients (by default they accumulate)
    optimizer.zero_grad()
    # Forward pass
    outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
    loss = outputs['loss']
    train_loss_set.append(loss.item())
    # Backward pass
    loss.backward()
    # Update parameters and take a step using the computed gradient
    optimizer.step()

    # Update the learning rate.
    scheduler.step()


    # Update tracking variables
    tr_loss += loss.item()
    nb_tr_examples += b_input_ids.size(0)
    nb_tr_steps += 1

  print("Train loss: {}".format(tr_loss/nb_tr_steps))

  # Validation

  # Put model in evaluation mode to evaluate loss on the validation set
  model.eval()

  # Tracking variables
  eval_loss, eval_accuracy = 0, 0
  nb_eval_steps, nb_eval_examples = 0, 0

  # Evaluate data for one epoch
  for batch in validation_dataloader:
    # Add batch to GPU
    batch = tuple(t.to(device) for t in batch)
    # Unpack the inputs from our dataloader
    b_input_ids, b_input_mask, b_labels = batch
    # Telling the model not to compute or store gradients, saving memory and speeding up validation
    with torch.no_grad():
      # Forward pass, calculate logit predictions
      logits = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)

    # Move logits and labels to CPU
    logits = logits['logits'].detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    tmp_eval_accuracy = flat_accuracy(logits, label_ids)

    eval_accuracy += tmp_eval_accuracy
    nb_eval_steps += 1

  print("Validation Accuracy: {}".format(eval_accuracy/nb_eval_steps))